# Agentic RCA and Context Learning
Inspect the governed investigation contract, diagnostic limits, reusable issue patterns, and approved RCA knowledge. Actual diagnostics execute inside the canonical workflow in notebook `08`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'config').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from dq_agent.config import load_app_config
from dq_agent.context_store import make_context_retriever
from dq_agent.workflow import InvestigationAction, InvestigationDecision

In [ ]:
config = load_app_config(ROOT)
store = make_context_retriever(config)
print('Context sync:', store.sync())
print('RCA limits:', {
    'maximum_depth': config.project.max_rca_rounds,
    'maximum_llm_calls': config.project.max_llm_calls_per_failure,
    'maximum_bytes_billed': config.project.query_limits.max_bytes_billed,
    'timeout_seconds': config.project.query_limits.timeout_seconds,
    'evidence_rows': config.project.query_limits.evidence_rows,
    'maximum_group_cardinality': config.project.query_limits.maximum_group_cardinality,
})

In [ ]:
issue_patterns = store.get_exact(context_type='issue_pattern')
learned = store.get_exact(origin='APPROVED_LEARNING')
display(pd.DataFrame([{
    'record_id': row['record_id'], 'subject': row['subject_key'],
    'diagnostics': row['payload'].get('diagnostics'), 'provenance': row['provenance'],
} for row in issue_patterns]))
print('Approved learned records:', len(learned))
display(pd.DataFrame(learned))

In [ ]:
example = InvestigationDecision(
    hypotheses=['Target date coverage may be incomplete'],
    enough_evidence=False, classification='undetermined', confidence=0.35,
    next_action=InvestigationAction(
        intent='date_coverage', side='both', columns=['business_date'],
        rationale='Compare bounded min/max coverage before drawing a conclusion',
        expected_evidence='Source and target minimum and maximum business dates',
    ),
)
display(example.model_dump())

The LLM returns this typed decision, not SQL. The workflow validates the intent and identifiers, compiles allowlisted read-only SQL, applies warehouse budgets, and returns evidence for the next decision. Confirmed or likely reusable conclusions still enter `learned_context.yaml` only after the Excel flow in notebook `02`.